In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("raw_dataset.csv")
FEATURES = [
    "Simulated_Moisture_Deficit_%",
    "Atmospheric_Temperature_C",
    "Atmospheric_Humidity_%",
]
PERCENTAGE_COLUMNS = [
    "Soil_Moisture_%",
    "Moisture_Before_Irrigation_%",
    "Simulated_Threshold_%",
    "Simulated_Moisture_Deficit_%",
    "Atmospheric_Humidity_%",
]
TARGET = "Simulated_Water_Delivered_L_per_m2"

raw = pd.read_csv(DATA_PATH)
required_columns = FEATURES + PERCENTAGE_COLUMNS + [TARGET, "Timestamp"]
missing_columns = sorted(set(required_columns) - set(raw.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

clean = raw.copy()
clean["Timestamp"] = pd.to_datetime(clean["Timestamp"], errors="coerce")
numeric_columns = sorted(set(FEATURES + PERCENTAGE_COLUMNS + [TARGET]))
for column in numeric_columns:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

rows_before = len(clean)
clean = clean.drop_duplicates()
clean = clean.dropna(subset=required_columns)
finite_values = np.isfinite(clean[numeric_columns]).all(axis=1)
valid_ranges = (
    clean[PERCENTAGE_COLUMNS].ge(0).all(axis=1)
    & clean[PERCENTAGE_COLUMNS].le(100).all(axis=1)
    & clean[TARGET].ge(0)
)
clean = clean.loc[finite_values & valid_ranges].sort_values("Timestamp").reset_index(drop=True)

print(f"Raw rows: {rows_before}; cleaned rows: {len(clean)}; removed: {rows_before - len(clean)}")
print("Model inputs:", FEATURES)
print("Target:", TARGET)
expected_deficit = (clean["Simulated_Threshold_%"] - clean["Moisture_Before_Irrigation_%"]).clip(lower=0)
deficit_mismatches = ~np.isclose(clean["Simulated_Moisture_Deficit_%"], expected_deficit, atol=0.011)
print(f"Deficit-formula mismatches (rounding tolerance 0.011): {int(deficit_mismatches.sum())}")
display(clean[FEATURES + [TARGET]].describe().round(3))
display(clean[FEATURES + [TARGET]].corr().round(3))

clean.to_csv("cleaned_dataset.csv", index=False)
print("Saved cleaned data to cleaned_dataset.csv")

Raw rows: 300; cleaned rows: 300; removed: 0
Model inputs: ['Simulated_Moisture_Deficit_%', 'Atmospheric_Temperature_C', 'Atmospheric_Humidity_%']
Target: Simulated_Water_Delivered_L_per_m2
Deficit-formula mismatches (rounding tolerance 0.011): 0


,Simulated_Moisture_Deficit_%,Atmospheric_Temperature_C,Atmospheric_Humidity_%,Simulated_Water_Delivered_L_per_m2
count,300.000,300.000,300.000,300.000
mean,8.717,27.442,78.571,3.381
std,4.900,2.799,7.525,1.973
min,0.000,21.340,62.410,0.000
25%,4.725,24.942,71.920,1.732
50%,8.785,27.430,78.225,3.303
75%,12.512,29.880,85.472,4.866
max,19.080,33.130,98.000,7.960


,Simulated_Moisture_Deficit_%,Atmospheric_Temperature_C,Atmospheric_Humidity_%,Simulated_Water_Delivered_L_per_m2
Simulated_Moisture_Deficit_%,1.000,0.500,-0.669,0.994
Atmospheric_Temperature_C,0.500,1.000,-0.887,0.547
Atmospheric_Humidity_%,-0.669,-0.887,1.000,-0.706
Simulated_Water_Delivered_L_per_m2,0.994,0.547,-0.706,1.000


Saved cleaned data to cleaned_dataset.csv


## ANFIS training and evaluation

The three inputs (soil-moisture deficit, atmospheric temperature, and atmospheric humidity) predict delivered water in L/m². Two Gaussian membership functions per input form eight first-order Sugeno rules. The final 20% of records are held out chronologically for evaluation, and the fitted parameters are saved to `anfis_model.npz`.

In [2]:
from itertools import product

from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if len(clean) < 20:
    raise ValueError("At least 20 cleaned observations are required to train and evaluate ANFIS.")

X = clean[FEATURES].to_numpy(dtype=float)
y = clean[TARGET].to_numpy(dtype=float)
split_at = int(len(clean) * 0.8)
X_train_raw, X_test_raw = X[:split_at], X[split_at:]
y_train, y_test = y[:split_at], y[split_at:]

input_mean = X_train_raw.mean(axis=0)
input_scale = X_train_raw.std(axis=0)
input_scale[input_scale == 0] = 1.0
X_train = (X_train_raw - input_mean) / input_scale
X_test = (X_test_raw - input_mean) / input_scale
X_train_augmented = np.column_stack((np.ones(len(X_train)), X_train))
n_features = X_train.shape[1]
n_memberships = 2
rules = np.asarray(list(product(range(n_memberships), repeat=n_features)), dtype=int)
n_rules = len(rules)

centers_initial = np.quantile(X_train, [0.25, 0.75], axis=0).T
sigma_initial = np.maximum(X_train.std(axis=0) * 0.75, 0.35)
log_sigmas_initial = np.log(np.repeat(sigma_initial[:, None], n_memberships, axis=1))
consequents_initial = np.zeros((n_rules, n_features + 1))
consequents_initial[:, 0] = y_train.mean()
initial_parameters = np.concatenate(
    (centers_initial.ravel(), log_sigmas_initial.ravel(), consequents_initial.ravel())
)

n_premise_parameters = n_features * n_memberships
n_consequent_parameters = n_rules * (n_features + 1)
ridge = 1e-4
consequent_penalty_mask = np.ones_like(consequents_initial)
consequent_penalty_mask[:, 0] = 0.0

def unpack_parameters(parameters):
    centers = parameters[:n_premise_parameters].reshape(n_features, n_memberships)
    log_sigmas = parameters[n_premise_parameters:2 * n_premise_parameters].reshape(n_features, n_memberships)
    consequents = parameters[2 * n_premise_parameters:].reshape(n_rules, n_features + 1)
    return centers, np.exp(log_sigmas), log_sigmas, consequents

def forward_anfis(inputs, centers, sigmas, consequents):
    log_firing_strengths = np.zeros((len(inputs), n_rules))
    for feature_index in range(n_features):
        membership_index = rules[:, feature_index]
        difference = inputs[:, feature_index, None] - centers[feature_index, membership_index]
        log_firing_strengths -= 0.5 * (difference / sigmas[feature_index, membership_index]) ** 2
    normalized_weights = np.exp(log_firing_strengths - logsumexp(log_firing_strengths, axis=1, keepdims=True))
    inputs_augmented = np.column_stack((np.ones(len(inputs)), inputs))
    rule_outputs = inputs_augmented @ consequents.T
    predictions = np.sum(normalized_weights * rule_outputs, axis=1)
    return predictions, normalized_weights, rule_outputs

def objective(parameters):
    centers, sigmas, log_sigmas, consequents = unpack_parameters(parameters)
    predictions, weights, rule_outputs = forward_anfis(X_train, centers, sigmas, consequents)
    residual = predictions - y_train
    loss = np.mean(residual ** 2) + ridge * np.sum((consequents * consequent_penalty_mask) ** 2)

    consequent_gradient = (2.0 / len(X_train)) * (weights * residual[:, None]).T @ X_train_augmented
    consequent_gradient += 2.0 * ridge * consequents * consequent_penalty_mask
    firing_gradient = (2.0 / len(X_train)) * residual[:, None] * weights * (rule_outputs - predictions[:, None])
    center_gradient = np.zeros_like(centers)
    log_sigma_gradient = np.zeros_like(log_sigmas)
    for feature_index in range(n_features):
        membership_index = rules[:, feature_index]
        difference = X_train[:, feature_index, None] - centers[feature_index, membership_index]
        sigma = sigmas[feature_index, membership_index]
        center_terms = firing_gradient * difference / (sigma ** 2)
        sigma_terms = firing_gradient * (difference / sigma) ** 2
        center_gradient[feature_index] = np.bincount(
            membership_index, weights=center_terms.sum(axis=0), minlength=n_memberships
        )
        log_sigma_gradient[feature_index] = np.bincount(
            membership_index, weights=sigma_terms.sum(axis=0), minlength=n_memberships
        )

    gradient = np.concatenate(
        (center_gradient.ravel(), log_sigma_gradient.ravel(), consequent_gradient.ravel())
    )
    return loss, gradient

parameter_bounds = (
    [(None, None)] * n_premise_parameters
    + [(np.log(0.05), np.log(5.0))] * n_premise_parameters
    + [(None, None)] * n_consequent_parameters
)
result = minimize(
    objective,
    initial_parameters,
    method="L-BFGS-B",
    jac=True,
    bounds=parameter_bounds,
    options={"maxiter": 5000, "ftol": 1e-10, "maxls": 40},
)
if not result.success:
    print(f"ANFIS optimizer stopped before convergence: {result.message}")

premise_centers, premise_sigmas, _, consequent_parameters = unpack_parameters(result.x)

def predict_anfis(inputs_raw):
    standardized_inputs = (inputs_raw - input_mean) / input_scale
    predictions, _, _ = forward_anfis(standardized_inputs, premise_centers, premise_sigmas, consequent_parameters)
    return np.maximum(predictions, 0.0)

y_train_pred = predict_anfis(X_train_raw)
y_test_pred = predict_anfis(X_test_raw)
metrics = pd.DataFrame(
    {
        "MAE (L/m²)": [mean_absolute_error(y_train, y_train_pred), mean_absolute_error(y_test, y_test_pred)],
        "RMSE (L/m²)": [np.sqrt(mean_squared_error(y_train, y_train_pred)), np.sqrt(mean_squared_error(y_test, y_test_pred))],
        "R²": [r2_score(y_train, y_train_pred), r2_score(y_test, y_test_pred)],
    },
    index=["Train", "Chronological test"],
)
display(metrics.round(4))
print(f"Trained {n_rules} Sugeno rules on {len(X_train)} records; evaluated on {len(X_test)} later records.")

np.savez(
    "anfis_model.npz",
    features=np.asarray(FEATURES),
    input_mean=input_mean,
    input_scale=input_scale,
    premise_centers=premise_centers,
    premise_sigmas=premise_sigmas,
    rules=rules,
    consequents=consequent_parameters,
)
print("Saved trained model parameters to anfis_model.npz")

,MAE (L/m²),RMSE (L/m²),R²
Train,0.1057,0.1495,0.9945
Chronological test,0.0899,0.1240,0.9943


Trained 8 Sugeno rules on 240 records; evaluated on 60 later records.
Saved trained model parameters to anfis_model.npz
